# Preliminary run of the base learners — NCHS perinatal model

Framework stage: **S8** (modeling). With 6,211 events per parameter, the framework rule (20 or more) sets as primary learner **the best base learner of a preliminary run**. This notebook is that run: it compares penalized logistic regression (LR), random forest (RF), extra trees (ET), XGBoost and a stacked ensemble, and applies the selection rule written before seeing results. Spanish version: `04_preliminary_learners_es.ipynb` (same code).

**Design.**
- Development years only (2016–2021); 2022–2023 are untouched (temporal test, C4).
- Sample: **all events** and **2% of non-events**, with weight 1/0.02 for non-events, so that AUPRC, calibration and the top 10% of risk refer to the full population. AUROC does not depend on this sampling.
- One stratified 80/20 split: each learner is fitted on the 80%; the 20% is used for comparison, with bootstrap intervals (300 replicates).
- **Hyperparameter search (widened on 2026-11-22):** random search with **the same budget for every learner**: 20 combinations per learner, drawn from ranges fixed before running, scored by AUROC in 3 inner folds of a stratified 30% subsample of the fit part; the best combination is refitted on the whole fit part. Ranges: LR, C from 0.0001 to 10 and L1 or L2 penalty; RF and ET, depth 8 to 30 or unlimited, minimum leaf 20 to 1,000 and features per split √p, 30% or 50% (300 trees); XGBoost, depth 3 to 10, learning rate 0.01 to 0.2, minimum child weight 1 to 100, row and column subsampling 0.6 to 1, L1 and L2 regularization, and number of trees by early stopping (up to 2,000). The output lists every combination tried and its AUROC.
- Missing values: **every learner, XGBoost included, imputes the median inside training** (decision D3). Correction of 2026-11-22: in the first run XGBoost used its own handling of missing values, which can send unknowns to their own branch and learn the type of certificate; that version is kept only as a **leakage diagnostic**, with the same hyperparameters, and cannot be selected.
- **Selection rule:** primary learner = the base learner with the highest AUROC; if the difference with a simpler one is under 0.005, the simpler one is preferred (LR, then XGBoost, then RF and ET). The ensemble is adopted only if it beats the primary by 0.005 or more with the 95% CI of the difference above 0.
- **Accuracy and confusion matrix** (section 4): not framework metrics; added on request. They need a threshold: the top 10% (and 5%) of risk is flagged. With a prevalence of 0.66%, high accuracy does not mean a good model: flagging nobody already gives 99.3%.

Output: `data/outputs_04_preliminary_learners.txt`. Requires `xgboost` (`%pip install xgboost`). On a desktop computer it takes in the order of 1.5 to 3 hours.

In [ ]:
import sys, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_04_preliminary_learners.txt"

SEED = 20261121
DEV_YEARS = list(range(2016, 2022))       # 2022-2023 are kept untouched for the temporal test (C4)
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))   # share of non-events sampled; all events kept
QUICK = os.environ.get("NCHS_QUICK") == "1"               # only to test the code on a small sample
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
PREDICTORS = ['age', 'race_black', 'race_hispanic', 'race_asian', 'race_other', 'educ_some_college',
              'educ_bachelor_plus', 'born_outside_us', 'wic', 'prior_living_1', 'prior_living_2plus',
              'prior_child_died', 'interval_lt18', 'interval_18_59', 'interval_60plus', 'prior_cesarean',
              'bmi', 'smoked_before_pregnancy', 'care_after_first_trimester', 'no_prenatal_care',
              'pregestational_diabetes', 'chronic_hypertension', 'infertility_treatment', 'multiple_pregnancy']
lines = []

def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

## 1. Development sample and split

In [ ]:
# Development sample: every event and a random FRACTION of non-events of 2016-2021, with weights
# 1/FRACTION for non-events so that weighted metrics and calibration refer to the full population.
rng = np.random.default_rng(SEED)
parts, n_dev, e_dev = [], 0, 0
for y in DEV_YEARS:
    c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS + ["label_main"])
    n_dev += len(c); e_dev += int(c["label_main"].sum())
    keep = c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)
    parts.append(c[keep])
s = pd.concat(parts, ignore_index=True); del parts
if QUICK:
    s = s.sample(n=min(len(s), 40000), random_state=SEED)
y = s.pop("label_main").to_numpy(bool)
w = np.where(y, 1.0, 1.0 / FRACTION)
X = s.astype("float32")

# Preliminary run (framework, S8): one stratified split of the development sample, 80% to fit and
# tune (inner 3-fold search), 20% to compare learners. Products are independent units (no mother id).
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=SEED)
show("Development data", pd.Series(dtype=object, data={
    "development products (2016-2021)": n_dev, "development events": e_dev,
    "prevalence": round(e_dev / n_dev, 5), f"sample: events + {100 * FRACTION:.0f}% of non-events": len(X),
    "fit/tune part": len(X_tr), "comparison part": len(X_te), "events in comparison part": int(y_te.sum())}))

## 2. Hyperparameter search, fitting of the base learners, the ensemble and the leakage diagnostic

In [ ]:
# Hyperparameter search (correction of 2026-11-22: the first runs tried only 2-3 values of one
# hyperparameter per learner). Random search with the SAME budget for every learner: N_CONFIGS
# combinations drawn from prespecified ranges, scored by AUROC in 3 inner folds of a stratified 30%
# subsample of the fit part; the best combination is then refitted on the whole fit part.
# Every learner imputes the median inside training (decision D3). XGBoost: number of trees chosen by
# early stopping (up to 2,000, patience 50) in each inner fold; the refit uses their mean.
from sklearn.base import clone
N_CONFIGS = 3 if QUICK else 20
SEARCH_SHARE = 0.30
rs = np.random.default_rng(SEED)
loguni = lambda lo, hi: float(np.exp(rs.uniform(np.log(lo), np.log(hi))))

def sample_config(name):
    if name == "LR":
        return {"C": loguni(1e-4, 10), "penalty": str(rs.choice(["l1", "l2"]))}
    if name in ("RF", "ET"):
        return {"max_depth": [8, 12, 16, 20, 30, None][rs.integers(6)],
                "min_samples_leaf": int(loguni(20, 1000)),
                "max_features": ["sqrt", 0.3, 0.5][rs.integers(3)]}
    return {"max_depth": int(rs.integers(3, 11)), "learning_rate": loguni(0.01, 0.2),
            "min_child_weight": loguni(1, 100), "subsample": float(rs.uniform(0.6, 1.0)),
            "colsample_bytree": float(rs.uniform(0.6, 1.0)), "reg_lambda": loguni(0.1, 10),
            "reg_alpha": loguni(1e-3, 1)}

def build(name, cfg, n_estimators=None):
    imp = SimpleImputer(strategy="median")
    if name == "LR":
        return make_pipeline(imp, StandardScaler(), LogisticRegression(solver="liblinear", max_iter=500, **cfg))
    n_trees = 50 if QUICK else 300
    if name == "RF":
        return make_pipeline(imp, RandomForestClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    if name == "ET":
        return make_pipeline(imp, ExtraTreesClassifier(n_estimators=n_trees, n_jobs=N_JOBS, random_state=SEED, **cfg))
    return make_pipeline(imp, XGBClassifier(n_estimators=n_estimators or (100 if QUICK else 2000), tree_method="hist",
                                            n_jobs=N_JOBS, random_state=SEED, eval_metric="auc", **cfg))

def fit_w(est, X_, y_, w_, **extra):
    step = est.steps[-1][0]
    return est.fit(X_, y_, **{f"{step}__sample_weight": w_, **{f"{step}__{k}": v for k, v in extra.items()}})

X_se, _, y_se, _, w_se, _ = train_test_split(X_tr, y_tr, w_tr, train_size=SEARCH_SHARE, stratify=y_tr, random_state=SEED)
inner = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_se, y_se))
fitted, pred, grid_rows, trials = {}, {}, [], []
for name in ["LR", "RF", "ET", "XGB"]:
    t = time.time(); best = (-1, None, None)
    for k in range(N_CONFIGS):
        cfg = sample_config(name); aucs, iters = [], []
        for tr, va in inner:
            est = build(name, cfg)
            if name == "XGB":
                imp = SimpleImputer(strategy="median").fit(X_se.iloc[tr])
                xgb = est.steps[-1][1].set_params(early_stopping_rounds=50)
                xgb.fit(imp.transform(X_se.iloc[tr]), y_se[tr], sample_weight=w_se[tr],
                        eval_set=[(imp.transform(X_se.iloc[va]), y_se[va])], verbose=False)
                p = xgb.predict_proba(imp.transform(X_se.iloc[va]))[:, 1]; iters.append(xgb.best_iteration + 1)
            else:
                p = fit_w(est, X_se.iloc[tr], y_se[tr], w_se[tr]).predict_proba(X_se.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y_se[va], p))
        score = float(np.mean(aucs))
        trials.append({"learner": name, "inner AUROC": score, **{k: (round(v, 4) if isinstance(v, float) else str(v)) for k, v in cfg.items()}})
        if score > best[0]:
            best = (score, cfg, int(np.mean(iters)) if iters else None)
    score, cfg, n_iter = best
    fitted[name] = fit_w(build(name, cfg, n_estimators=n_iter), X_tr, y_tr, w_tr)
    pred[name] = fitted[name].predict_proba(X_te)[:, 1]
    shown = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in cfg.items()}
    if n_iter: shown["n_estimators"] = n_iter
    grid_rows.append({"learner": name, "best combination": json.dumps(shown), "inner AUROC": round(score, 4),
                      "minutes": round((time.time() - t) / 60, 1)})
    print(name, "done in", grid_rows[-1]["minutes"], "min")
show(f"Random search ({N_CONFIGS} combinations per learner, 3 inner folds on {int(SEARCH_SHARE * 100)}% of the fit part)",
     pd.DataFrame(grid_rows).set_index("learner"))
show("Spread of the inner AUROC over the combinations tried (how much the search mattered)",
     pd.DataFrame(trials).groupby("learner")["inner AUROC"].agg(["min", "median", "max"]).round(4))
trials_df = pd.DataFrame(trials)
for name in ["LR", "RF", "ET", "XGB"]:
    t_ = trials_df[trials_df["learner"] == name].dropna(axis=1, how="all").drop(columns="learner")
    show(f"All combinations tried: {name}", t_.sort_values("inner AUROC", ascending=False).round(4).reset_index(drop=True))

# Stacked ensemble with a logistic meta-learner (comparison only): out-of-fold predictions of each
# tuned base learner inside the fit part train the meta-learner; the base learners fitted on the
# whole fit part score the comparison part.
t = time.time()
oof = np.zeros((len(X_tr), len(fitted)))
for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_tr, y_tr):
    for j, est in enumerate(fitted.values()):
        m = fit_w(clone(est), X_tr.iloc[tr], y_tr[tr], w_tr[tr])
        oof[va, j] = m.predict_proba(X_tr.iloc[va])[:, 1]
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
meta = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # meta-learner on the logit scale
meta.fit(logit(oof), y_tr, logisticregression__sample_weight=w_tr)
pred["Stack"] = meta.predict_proba(logit(np.column_stack([pred[k] for k in fitted])))[:, 1]
print("Stack done in", round((time.time() - t) / 60, 1), "min")

# Leakage diagnostic (not eligible as primary): XGBoost with its NATIVE handling of missing values,
# which can route unknown values to their own branch and so learn the type of certificate.
native = clone(fitted["XGB"].steps[-1][1]).fit(X_tr, y_tr, sample_weight=w_tr)   # same tuned hyperparameters
pred["XGB native missing (diagnostic)"] = native.predict_proba(X_te)[:, 1]

## 3. Comparison and selection of the primary learner; 4. accuracy and confusion matrices

In [ ]:
# Comparison on the 20% part. AUROC does not depend on sampling the non-events; AUPRC and the
# observed/expected ratio use the weights so they refer to the full population.
B = 50 if QUICK else 300
boot = np.random.default_rng(SEED)
idx_pos, idx_neg = np.where(y_te)[0], np.where(~y_te)[0]
boot_idx = [np.concatenate([boot.choice(idx_pos, len(idx_pos)), boot.choice(idx_neg, len(idx_neg))]) for _ in range(B)]

def auroc(p, ix=None):
    ix = slice(None) if ix is None else ix
    return roc_auc_score(y_te[ix], p[ix])

rows = {}
for name, p in pred.items():
    a = [auroc(p, ix) for ix in boot_idx]
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    cut = p[o][np.searchsorted(cw, 0.90 * cw[-1])]          # weighted 90th percentile of risk
    top = p >= cut
    rows[name] = {"AUROC": round(auroc(p), 4), "AUROC 2.5%": round(np.percentile(a, 2.5), 4),
                  "AUROC 97.5%": round(np.percentile(a, 97.5), 4),
                  "AUPRC (weighted)": round(average_precision_score(y_te, p, sample_weight=w_te), 4),
                  "observed/expected": round(float(y_te.sum() / np.sum(w_te * p)), 3),
                  "events in top 10% (%)": round(100 * float(y_te[top].sum() / y_te.sum()), 1)}
res = pd.DataFrame(rows).T
show(f"Preliminary comparison on the 20% part (prevalence {y_te.sum() / w_te.sum():.5f})", res)

# Prespecified selection rule (outcome card, S8): primary = best base learner by AUROC; when the
# difference with a simpler learner is under 0.005, the simpler one is preferred (LR < XGB < RF/ET).
order = ["LR", "XGB", "RF", "ET"]
best = max(order, key=lambda k: res.loc[k, "AUROC"])
primary = next(k for k in order if res.loc[best, "AUROC"] - res.loc[k, "AUROC"] < 0.005)
d_stack = [auroc(pred["Stack"], ix) - auroc(pred[primary], ix) for ix in boot_idx]
lo, hi = np.percentile(d_stack, [2.5, 97.5])
adopt = (res.loc["Stack", "AUROC"] - res.loc[primary, "AUROC"] >= 0.005) and lo > 0
show("Selection", pd.Series({"best base learner by AUROC": best, "primary learner (rule)": primary,
                             "Stack - primary AUROC": f"{np.mean(d_stack):+.4f} ({lo:+.4f} to {hi:+.4f})",
                             "ensemble adopted": adopt}))
# Accuracy and confusion matrices (requested in addition to the framework metrics). A threshold is
# needed: the product is flagged when its risk is in the top 10% (and, second, the top 5%) of the
# weighted risk distribution, the same cut the framework uses for the share of events. Counts are
# weighted, so they estimate the full population of the comparison part (about 20% of 2016-2021).
def wcut(p, share):
    o = np.argsort(p); cw = np.cumsum(w_te[o])
    return p[o][np.searchsorted(cw, (1 - share) * cw[-1])]

cm_rows, prev_te = [], y_te.sum() / w_te.sum()
for share in (0.10, 0.05):
    for name, p in pred.items():
        flag = p >= wcut(p, share)
        tp = float(w_te[flag & y_te].sum()); fn = float(w_te[~flag & y_te].sum())
        fp = float(w_te[flag & ~y_te].sum()); tn = float(w_te[~flag & ~y_te].sum())
        n = tp + fn + fp + tn
        cm_rows.append({"flagged": f"top {int(share * 100)}%", "learner": name,
                        "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
                        "accuracy": round((tp + tn) / n, 4), "sensitivity": round(tp / (tp + fn), 3),
                        "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4),
                        "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrices and accuracy (weighted to the full population of the comparison part)",
     pd.DataFrame(cm_rows).set_index(["flagged", "learner"]))
show("Reference: accuracy of flagging nobody (all predicted without event)", round(1 - prev_te, 4))
OUT_FILE.write_text("Preliminary run of the base learners (perinatal model, development 2016-2021)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")